# 03 - EDA and Feature Engineering**Objective:** Explore Iris data patterns and create new features.**Steps:**1. Statistical summary and class distribution2. Visualizations (distributions, correlations, feature comparison by species)3. Feature scaling4. Feature engineering (interaction and ratio features)5. Save engineered data

In [ ]:
import pandas as pdimport numpy as npfrom pathlib import Pathimport matplotlib.pyplot as pltimport seaborn as snssns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)print("Libraries imported successfully")

In [ ]:
# TODO: Load the clean data and inspect its structure# Start by reading data/processed/clean_data.csv into a DataFrame.# Then use .info() to see column dtypes and non-null counts,# .head() to preview a few rows, and .describe() for summary statistics.# The dataset has 150 rows, 4 numeric features, and a 3-class target.PROCESSED_DIR = Path("../data/processed")# df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")# print(f"Shape: {df.shape}")# print(df.info())# print(df.head())# print(df.describe())

### Statistical Summary & DistributionBefore building any models, it is important to understand the range and spread of your data.The target variable — `class` — has three categories (a multi-class classification problem).All three species have exactly 50 samples, so the dataset is perfectly balanced.Iris has 4 continuous features measuring sepal and petal dimensions in centimeters.

In [ ]:
# TODO: Examine the target variable distribution and feature summary# Use df.describe() to see min, max, mean, and quartiles for all 4 features.# Check the class distribution with df["class"].value_counts() — each species has 50 rows.## print(df.describe())# print(df["class"].value_counts())

#### A little primer on groupby - `groupby` is a powerful pandas method that allows you to split your data into groups based on some criteria, apply a function to each group, and then combine the results. For example, to see how average petal length differs between species, you can do:```pythonpetal_by_species = df.groupby("class")["petal_length"].mean()```- `aggregate` is a method that allows you to apply multiple functions to your grouped data. For example, to get both the mean and standard deviation of sepal length by species, you can do:```pythonstats_by_species = df.groupby("class")["sepal_length"].aggregate(["mean", "std"])```Aggregate functions can be any function that takes a Series and returns a single value, such as `mean`, `std`, `min`, `max`, etc.Aggregate can be deployed on multiple columns at once, and you can specify different functions for each column if needed.

In [ ]:
# === Executed Example: GroupBy and Aggregate ===# Small inline dataset showing how groupby splits data by class# and compares flower measurements between species.import pandas as pddata = pd.DataFrame({    "class": [0, 0, 0, 1, 1, 1],    "sepal_length": [5.1, 4.9, 5.0, 6.0, 6.2, 5.9],    "petal_length": [1.4, 1.5, 1.3, 4.5, 4.7, 4.4],})mean_by_class = data.groupby("class")["petal_length"].mean()print("Average petal length by class:\n", mean_by_class)stats_by_class = data.groupby("class")["sepal_length"].agg(["mean", "std", "min", "max"])print("\nSepal length statistics by class:\n", stats_by_class)

In [ ]:
# === Commented Template: GroupBy and Aggregate ===# Uncomment and adapt to your own dataset.# import pandas as pd# data = pd.DataFrame({#     "group_col": [val1, val1, val2, val2],#     "value_col": [10, 20, 30, 40],# })# grouped = data.groupby("group_col")["value_col"].mean()# stats = data.groupby("group_col")["value_col"].agg(["mean", "std", "min", "max"])

In [ ]:
# TODO: Compare feature distributions by species# Which measurements best separate the three species?# Use boxplots to compare all 4 features.# Iris-setosa should separate clearly on petal dimensions.## features_to_plot = ["sepal_length", "sepal_width", "petal_length", "petal_width"]# fig, axes = plt.subplots(2, 2, figsize=(12, 10))# for ax, feature in zip(axes.ravel(), features_to_plot):#     sns.boxplot(x="class", y=feature, data=df, ax=ax)#     ax.set_title(f"{feature} by class")# plt.tight_layout()# plt.show()

### VisualizationsVisual exploration helps you spot patterns and relationships that summary statistics miss.Focus on:- How each feature is distributed (histograms)- Which features correlate with each other and with the target (heatmap)- Which features separate the three species (boxplots)

In [ ]:
# TODO: Plot histograms for all numerical features# Iris has only 4 features, so plot all of them.# Use df[features].hist() with bins=15 and a large figure size.# features_to_plot = ["sepal_length", "sepal_width", "petal_length", "petal_width"]# df[features_to_plot].hist(bins=15, figsize=(12, 8))# plt.tight_layout()# plt.show()

In [ ]:
# TODO: Create a correlation heatmap# Use df.corr() to compute pairwise correlations between numeric columns.# With only 4 features + the target, the full heatmap is easy to read.# Then visualize with sns.heatmap().## TODO: Identify the features most correlated with species# Sort the correlation values for the class column to see which features# best distinguish the three species.# plt.figure(figsize=(8, 6))# sns.heatmap(df.corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")# plt.title("Feature Correlation Heatmap")# plt.show()## target_corr = df.corr()["class"].sort_values(ascending=False)# print("Correlations with class:")# print(target_corr)

### Feature ScalingMany machine learning algorithms (SVM, logistic regression, neural networks) are sensitive tothe scale of input features. StandardScaler transforms each feature to have mean 0 andstandard deviation 1, which puts all features on equal footing.Tree-based models (Random Forest, XGBoost) do not require scaling since they split onthresholds independently of feature magnitude.

In [ ]:
# === Executed Example: Feature Scaling ===# Small inline dataset showing how StandardScaler transforms features# to have mean ~0 and std ~1.from sklearn.preprocessing import StandardScalerimport pandas as pddata = pd.DataFrame({    "sepal_length": [5.1, 4.9, 5.0, 6.0, 6.2],    "petal_length": [1.4, 1.5, 1.3, 4.5, 4.7],    "class": [0, 0, 0, 1, 1],})scaler = StandardScaler()scaled_features = scaler.fit_transform(data[["sepal_length", "petal_length"]])scaled_df = pd.DataFrame(scaled_features, columns=["sepal_length_scaled", "petal_length_scaled"])scaled_df["class"] = data["class"]print(scaled_df)print(f"Means after scaling: {scaled_df[['sepal_length_scaled', 'petal_length_scaled']].mean().values}")print(f"Stds after scaling: {scaled_df[['sepal_length_scaled', 'petal_length_scaled']].std().values}")

In [ ]:
from sklearn.preprocessing import StandardScaler# TODO: Scale the 4 numeric features# Iris has only continuous features, all measured in cm.# StandardScaler normalises them so SVM / logistic regression aren't biased# by the absolute magnitude of each measurement.## Create a StandardScaler, call fit_transform() to compute the mean and std# and return the scaled array in one step.## After scaling, verify that each feature has mean ~0 and std ~1.# numeric_features = ["sepal_length", "sepal_width", "petal_length", "petal_width"]# scaler = StandardScaler()# df_scaled = scaler.fit_transform(df[numeric_features])## Rename the scaled columns with a "_scaled" suffix# df_scaled = pd.DataFrame(df_scaled, columns=[f"{col}_scaled" for col in numeric_features])# print(f"Means after scaling: {df_scaled.mean().values}")# print(f"Stds after scaling: {df_scaled.std().values}")# print(f"All means near zero: {np.allclose(df_scaled.mean(), 0, atol=1e-10)}")

### Feature EngineeringNew features derived from existing columns can capture interactions and non-linear relationships.Good candidates for Iris:- **Petal area**: `petal_length × petal_width` — captures overall petal size- **Sepal area**: `sepal_length × sepal_width` — overall sepal size- **Petal-to-sepal ratio**: `petal_length / sepal_length` — relative flower proportionBe careful with division by zero — add a small epsilon or +1 to the denominator.#### NoteIn pandas, you can create interaction features like this:```pythondf["feature1_feature2"] = df["feature1"] * df["feature2"]```

In [ ]:
# === Executed Example: Interaction Features ===# Multiplication and ratio on a small inline dataset.import pandas as pddata = pd.DataFrame({    "sepal_length": [5.1, 4.9, 6.0, 6.2, 5.9],    "sepal_width": [3.5, 3.0, 3.3, 2.9, 3.0],    "petal_length": [1.4, 1.5, 4.5, 4.7, 4.4],    "petal_width": [0.2, 0.2, 1.5, 1.6, 1.4],})data["petal_area"] = data["petal_length"] * data["petal_width"]print("Petal area:\n", data[["petal_length", "petal_width", "petal_area"]])data["petal_sepal_ratio"] = data["petal_length"] / data["sepal_length"]print("\nPetal / Sepal ratio:\n", data[["petal_length", "sepal_length", "petal_sepal_ratio"]])

In [ ]:
# === Commented Template: Interaction Features ===# Uncomment and adapt to your own dataset.# import pandas as pd# data = pd.DataFrame({#     "feature_a": [val1, val2, val3],#     "feature_b": [val1, val2, val3],# })# data["a_times_b"] = data["feature_a"] * data["feature_b"]# EPS = 1e-6# data["a_over_b"] = data["feature_a"] / (data["feature_b"] + EPS)

In [ ]:
# TODO: Create new features based on flower morphology# The Iris dataset measures sepal and petal dimensions. Area and ratio# features can make species separation more pronounced.## Examples:#   df["petal_area"] = df["petal_length"] * df["petal_width"]#   df["sepal_area"] = df["sepal_length"] * df["sepal_width"]#   df["petal_sepal_ratio"] = df["petal_length"] / df["sepal_length"]## TODO: Verify the new features# Check that they have finite values and reasonable ranges with .describe().

In [ ]:
# TODO: Save the engineered data for the next notebook# Include both the original and new features.PROCESSED_DIR = Path("../data/processed")PROCESSED_DIR.mkdir(parents=True, exist_ok=True)# df.to_csv(PROCESSED_DIR / "engineered_data.csv", index=False)# print("Engineered data saved to data/processed/engineered_data.csv")

### Exercises1. **Try different scalers**: Replace `StandardScaler` with `MinMaxScaler` or `RobustScaler` and compare how the scaled distributions look.2. **Ratio by class**: Use `df.groupby("class")` on `petal_sepal_ratio` to see if the ratio alone can separate all three species. Which species has the largest ratio?3. **Log transform features**: Petal dimensions are not heavily skewed for Iris, but try `np.log1p()` on `petal_area` anyway. How does the distribution change?4. **Pairplot**: Use `sns.pairplot()` on all 4 original features, coloring by class — do the three species form distinct clusters? Which two features separate them best?5. **Redundant features**: Check if `petal_length` and `petal_width` are highly correlated. If you had to drop one, which would you keep and why?